# Exercise 8: Comparing ERA5 Reanalysis with Climate DT Projections

In this advanced exercise, you'll:
1. Retrieve a historical ERA5 reanalysis field from CDS
2. Retrieve a Climate DT projection field
3. Regrid both to a common grid
4. Plot them side-by-side and compute the difference

This demonstrates how to compare present-day observations with future projections.

## Authentication

In [ ]:
%%capture cap
%run ../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.regrid
import numpy as np
import matplotlib.pyplot as plt
import os

## Live Request Toggle

In [ ]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

## Step 1: Retrieve ERA5 Reanalysis Data

We'll retrieve 2m temperature from ERA5 for a historical date using the CDS (Climate Data Store) source.

**Note**: You need a CDS API key configured (`~/.cdsapirc`) to access ERA5 data.

**EXERCISE**: Change the date/time to match a specific historical event you're interested in.

In [ ]:
request_era5 = {
    "product_type": "reanalysis",
    "variable": "2m_temperature",
    "year": "2020",
    "month": "01",
    "day": "02",
    "time": "12:00",
    "area": [75, -30, 25, 45],  # N, W, S, E (Europe)
    "grid": [0.25, 0.25],
}

era5_file = "../climate-dt/data/era5-training.grib"

if LIVE_REQUEST:
    data_era5 = earthkit.data.from_source(
        "cds", 
        "reanalysis-era5-single-levels",
        request_era5,
    )
    data_era5.to_target("file", era5_file)
else:
    data_era5 = earthkit.data.from_source("file", era5_file)

data_era5.ls()

## Step 2: Retrieve Climate DT Projection Data

Now we retrieve the same variable (2m temperature) from the Climate DT for a future date.

We use server-side interpolation to get the data on a regular grid that matches ERA5.

In [ ]:
request_climate = {
    "activity": "projections",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20400102",
    "time": "1200",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "2",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "167",
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "type": "fc",
}

climate_file = "../climate-dt/data/climate-dt-comparison-training.grib"

if LIVE_REQUEST:
    data_climate = earthkit.data.from_source(
        "polytope", "destination-earth", request_climate,
        address="polytope.mn5.apps.dte.destination-earth.eu",
        stream=False,
    )
    data_climate.to_target("file", climate_file)
else:
    data_climate = earthkit.data.from_source("file", climate_file)

data_climate.ls()

## Step 3: Regrid to a Common Grid

To compute differences, both datasets must be on the same grid. We'll regrid both to a common 1° × 1° regular lat-lon grid.

**EXERCISE**: Complete the regridding for both datasets.

In [ ]:
common_grid = {"grid": [1, 1]}

# Regrid ERA5 (from 0.25° to 1°)
era5_regridded = earthkit.regrid.interpolate(
    data_era5, out_grid=common_grid, method="linear"
)

# EXERCISE: Regrid Climate DT (from HEALPix to 1° regular)
# Hint: Use the same earthkit.regrid.interpolate pattern
climate_regridded = ___  # YOUR CODE HERE

print(f"ERA5 regridded fields: {len(era5_regridded)}")
print(f"Climate DT regridded fields: {len(climate_regridded)}")

## Step 4: Convert to xarray for Analysis

In [ ]:
ds_era5 = era5_regridded.to_xarray()
ds_climate = climate_regridded.to_xarray()

print("ERA5 shape:", ds_era5["2t"].shape)
print("Climate DT shape:", ds_climate["2t"].shape)

## Step 5: Plot Side-by-Side Comparison

**EXERCISE**: Complete the difference calculation and the third subplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5), subplot_kw={"projection": None})

# Get temperature arrays in Celsius
era5_temp = ds_era5["2t"].isel(time=0).values - 273.15
climate_temp = ds_climate["2t"].isel(time=0).values - 273.15

# EXERCISE: Calculate the difference (Climate DT - ERA5)
# This shows the projected warming
difference = ___  # YOUR CODE HERE

# Plot ERA5
ax1 = axes[0]
im1 = ax1.imshow(era5_temp, cmap="RdYlBu_r", vmin=-30, vmax=35, 
                  extent=[-30, 45, 25, 75], aspect="auto", origin="upper")
ax1.set_title("ERA5 Reanalysis (Jan 2020)")
plt.colorbar(im1, ax=ax1, label="\u00b0C")

# Plot Climate DT
ax2 = axes[1]
im2 = ax2.imshow(climate_temp, cmap="RdYlBu_r", vmin=-30, vmax=35,
                  extent=[-30, 45, 25, 75], aspect="auto", origin="upper")
ax2.set_title("Climate DT SSP3-7.0 (Jan 2040)")
plt.colorbar(im2, ax=ax2, label="\u00b0C")

# EXERCISE: Plot the difference in the third panel
# Use a diverging colormap (e.g., "RdBu_r") centered on 0
ax3 = axes[2]
# YOUR CODE HERE - plot the difference using ax3.imshow(...)
# Add a colorbar and title "Projected Change (\u00b0C)"


plt.tight_layout()
plt.show()

## Step 6: Statistical Summary

**EXERCISE**: Complete the statistical analysis of the difference field.

In [ ]:
# EXERCISE: Fill in the statistics
print("=== Projected Temperature Change (2040 vs 2020) ===")
print(f"Mean change: {___:.2f} \u00b0C")     # Hint: np.nanmean(difference)
print(f"Max warming: {___:.2f} \u00b0C")     # Hint: np.nanmax(difference)
print(f"Min change:  {___:.2f} \u00b0C")     # Hint: np.nanmin(difference)
print(f"Std dev:     {___:.2f} \u00b0C")     # Hint: np.nanstd(difference)

## Bonus: Regional Analysis

Try subsetting to a specific region (e.g., Mediterranean, Scandinavia) and computing the average warming for that region.

**EXERCISE**: Define latitude/longitude bounds for your region of interest and compute the mean temperature change.

In [ ]:
# EXERCISE: Define your region of interest
# Example: Mediterranean (lat: 30-45, lon: -5 to 35)

lat_min, lat_max = ___, ___  # YOUR REGION
lon_min, lon_max = ___, ___  # YOUR REGION

# Subset and compute regional mean
# YOUR CODE HERE

# print(f"Regional mean warming: {regional_mean:.2f} \u00b0C")

## Summary

In this exercise you've learned to:
- Retrieve data from multiple sources (CDS for ERA5, Polytope for Climate DT)
- Regrid heterogeneous datasets to a common grid
- Compute and visualize differences between reanalysis and projections
- Perform basic climate change signal analysis

Key takeaways:
- Always regrid to a common grid before comparing datasets
- The Climate DT provides km-scale climate projections that can be directly compared with ERA5
- SSP3-7.0 shows significant warming signals even for the 2040 timeframe